# EfficientNet-B4 - Train From Scratch trên Kaggle

Notebook này huấn luyện `EfficientNet-B4` từ đầu cho bài toán phân loại ảnh X-quang phổi hai lớp `NORMAL` và `PNEUMONIA` bằng PyTorch.

Điểm khác với fine-tuning:
- Không dùng pretrained ImageNet.
- Model khởi tạo trọng số ngẫu nhiên bằng `weights=None`.
- Thời gian train lâu hơn và accuracy ban đầu thường thấp hơn fine-tuning.

Kết quả gồm checkpoint tốt nhất, loss/accuracy/AUC curves, confusion matrix, ROC curve và ảnh dự đoán mẫu.

**Cách chạy trên Kaggle:** Add Input dataset đã chia sẵn `dataset_clean_70_15_15`, sau đó bấm `Run All`. Sau khi train xong nhớ bấm **Save Version → Save & Run All** để Kaggle giữ file model trong Output.


## 1. Cấu hình Kaggle

Code sẽ tự tìm thư mục `dataset_clean_70_15_15` trong `/kaggle/input`. Checkpoint và biểu đồ sẽ được lưu vào `/kaggle/working/outputs/efficientnet_b4_scratch` để Kaggle có thể giữ lại trong Output khi Save Version.


In [ ]:
import json
from pathlib import Path

# =========================
# CẤU HÌNH DATASET CHO KAGGLE
# =========================
# Trên Kaggle, bạn cần bấm Add Input và thêm dataset có tên dataset_clean.
# Cấu trúc bạn nói:
# /kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15/train
# /kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15/val
# /kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15/test
#
# Lưu ý: Kaggle đôi khi đổi dấu "_" thành "-", nên code dưới sẽ tự tìm cả
# dataset_clean và dataset-clean.

SPLIT_SEED = 42
CLASSES = ('NORMAL', 'PNEUMONIA')
DATASET_FOLDER_NAME = 'dataset_clean_70_15_15'


def prepared_dataset_is_complete(data_dir):
    """Kiểm tra dataset đã có đủ train/val/test và hai lớp NORMAL, PNEUMONIA."""
    data_dir = Path(data_dir)
    if not data_dir.exists():
        return False

    for split in ('train', 'val', 'test'):
        for class_name in CLASSES:
            folder = data_dir / split / class_name
            if not folder.exists():
                return False

            image_count = sum(
                1 for path in folder.iterdir()
                if path.suffix.lower() in {'.jpeg', '.jpg', '.png', '.bmp', '.webp'}
            )
            if image_count == 0:
                return False

    return True


def find_prepared_dataset():
    """Tự tìm dataset_clean_70_15_15 trên Kaggle/Colab/local."""
    candidates = [
        # Kaggle - trường hợp dataset title giữ nguyên dấu _
        Path('/kaggle/input/dataset_clean/dataset/dataset_clean_70_15_15'),
        Path('/kaggle/input/dataset_clean/dataset_clean_70_15_15'),

        # Kaggle - trường hợp Kaggle đổi dataset_clean thành dataset-clean
        Path('/kaggle/input/dataset-clean/dataset/dataset_clean_70_15_15'),
        Path('/kaggle/input/dataset-clean/dataset_clean_70_15_15'),

        # Colab/local cũ
        Path('/content/drive/MyDrive/CNN_ViT/dataset_clean_70_15_15'),
        Path('/content/drive/MyDrive/dataset_clean_70_15_15'),
        Path('/content/dataset_clean_70_15_15'),
        Path.cwd() / 'dataset' / 'dataset_clean_70_15_15',
        Path.cwd() / 'dataset_clean_70_15_15',
    ]

    for candidate in candidates:
        if prepared_dataset_is_complete(candidate):
            return candidate

    # Quét rộng trong /kaggle/input để tránh sai tên slug dataset.
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for candidate in kaggle_input.rglob(DATASET_FOLDER_NAME):
            if prepared_dataset_is_complete(candidate):
                return candidate

    raise FileNotFoundError(
        'Không tìm thấy dataset hợp lệ. Hãy kiểm tra bạn đã Add Input trên Kaggle chưa.\n'
        'Cấu trúc cần có dạng:\n'
        '/kaggle/input/<ten-dataset>/dataset/dataset_clean_70_15_15/train/NORMAL\n'
        '/kaggle/input/<ten-dataset>/dataset/dataset_clean_70_15_15/train/PNEUMONIA\n'
        'và tương tự cho val/test.'
    )


PREPARED_DATA_DIR = find_prepared_dataset()

# Kaggle không cho ghi vào /kaggle/input, nên lưu model/ảnh kết quả vào /kaggle/working.
BASE_OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else PREPARED_DATA_DIR.parent

print(f'Sử dụng dataset tại: {PREPARED_DATA_DIR.resolve()}')
print(f'Lưu kết quả tại: {BASE_OUTPUT_DIR.resolve()}')

# Hỗ trợ cả split_summary.json và split_sumary.json nếu file bị đặt tên thiếu chữ m.
for summary_name in ('split_summary.json', 'split_sumary.json'):
    summary_path = PREPARED_DATA_DIR / summary_name
    if summary_path.exists():
        with open(summary_path, 'r', encoding='utf-8') as file:
            print(json.dumps(json.load(file), indent=4, ensure_ascii=False))
        break

In [ ]:
from pathlib import Path
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

SEED = 42
DATA_DIR = PREPARED_DATA_DIR
OUTPUT_DIR = BASE_OUTPUT_DIR / 'outputs' / 'efficientnet_b4_scratch'
IMAGE_SIZE = 380
PREPROCESS_VERSION = 'center_crop_v1'
BATCH_SIZE = 8  # Giam xuong 4 neu GPU bi het bo nho.
EPOCHS = 25  # Train from scratch cần nhiều epoch hơn fine-tuning.
LEARNING_RATE = 3e-4  # LR cao hơn vì train từ đầu, có thể giảm còn 1e-4 nếu loss dao động.
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Doi thanh 2 trong Colab neu muon nap du lieu nhanh hon.
PATIENCE = 6
FREEZE_BACKBONE = False  # Train from scratch thì không freeze backbone.

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
print(f'Dataset path: {DATA_DIR.resolve()}')
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Khong tim thay dataset tai: {DATA_DIR.resolve()}')


## 2. Đọc dữ liệu, biến đổi ảnh và xem phân bố lớp

Vì train from scratch nên model không dùng trọng số ImageNet. Ảnh X-quang vẫn được chuyển thành 3 kênh để phù hợp đầu vào EfficientNet-B4, resize/crop về 380x380, rồi normalize đơn giản về khoảng gần `[-1, 1]`.


In [ ]:
NORMALIZE_MEAN = [0.5, 0.5, 0.5]
NORMALIZE_STD = [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(IMAGE_SIZE, antialias=True),
    transforms.CenterCrop((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(IMAGE_SIZE, antialias=True),
    transforms.CenterCrop((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

train_dataset = datasets.ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset = datasets.ImageFolder(DATA_DIR / 'val', transform=eval_transform)
test_dataset = datasets.ImageFolder(DATA_DIR / 'test', transform=eval_transform)
class_names = train_dataset.classes
num_classes = len(class_names)

for split_name, split_dataset in [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]:
    counts = np.bincount(split_dataset.targets, minlength=num_classes)
    print(f'{split_name:5s}: ' + ', '.join(f'{name}={count}' for name, count in zip(class_names, counts)))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (split_name, split_dataset) in zip(axes, [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]):
    counts = np.bincount(split_dataset.targets, minlength=num_classes)
    ax.bar(class_names, counts, color=['#9ecae1', '#3182bd'])
    ax.set_title(split_name)
    ax.set_ylabel('So anh')
plt.suptitle('Phan bo du lieu theo lop')
plt.tight_layout()
plt.show()

pin_memory = DEVICE.type == 'cuda'
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)


## 3. Khởi tạo EfficientNet-B4 từ đầu và các hàm huấn luyện

Quan trọng: dòng `models.efficientnet_b4(weights=None)` nghĩa là **không lấy model có sẵn**, toàn bộ trọng số được khởi tạo ngẫu nhiên.


In [ ]:
model = models.efficientnet_b4(weights=None)
if FREEZE_BACKBONE:
    for parameter in model.features.parameters():
        parameter.requires_grad = False
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(DEVICE)

positive_class_idx = class_names.index('PNEUMONIA')
train_counts = np.bincount(train_dataset.targets, minlength=num_classes)
class_weights = len(train_dataset) / (num_classes * train_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print(f'Classes: {class_names}')
print(f'Class weights: {class_weights.detach().cpu().numpy()}')
print('Training mode: FROM SCRATCH - weights=None')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    running_loss = 0.0
    labels_all, probs_all, predictions_all = [], [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            if is_training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * images.size(0)
        probabilities = torch.softmax(logits, dim=1)
        positive_probabilities = probabilities[:, positive_class_idx]
        predictions = probabilities.argmax(dim=1)
        labels_all.extend(labels.detach().cpu().numpy())
        probs_all.extend(positive_probabilities.detach().cpu().numpy())
        predictions_all.extend(predictions.detach().cpu().numpy())

    labels_all = np.asarray(labels_all)
    probs_all = np.asarray(probs_all)
    predictions = np.asarray(predictions_all)
    return {
        'loss': running_loss / len(loader.dataset),
        'accuracy': accuracy_score(labels_all, predictions),
        'auc': roc_auc_score(labels_all, probs_all),
        'labels': labels_all,
        'probs': probs_all,
        'predictions': predictions,
    }


## 4. Huan luyen va luu checkpoint tot nhat

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'train_auc': [], 'val_auc': []}
best_val_auc = -1.0
best_state = None
epochs_without_improvement = 0
checkpoint_path = OUTPUT_DIR / 'best_efficientnet_b4_scratch.pth'
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, optimizer)
    val_metrics = run_epoch(model, val_loader, criterion)
    scheduler.step(val_metrics['auc'])
    for metric in ['loss', 'accuracy', 'auc']:
        history[f'train_{metric}'].append(train_metrics[metric])
        history[f'val_{metric}'].append(val_metrics[metric])

    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f} auc={train_metrics['auc']:.4f} | val loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.4f} auc={val_metrics['auc']:.4f}")
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        best_state = copy.deepcopy(model.state_dict())
        torch.save({'model_state_dict': best_state, 'class_names': class_names, 'val_auc': float(best_val_auc), 'preprocess_version': PREPROCESS_VERSION, 'data_dir': DATA_DIR.name, 'training_mode': 'from_scratch', 'pretrained': False}, checkpoint_path)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping: validation AUC khong cai thien.')
            break

elapsed_minutes = (time.time() - start_time) / 60
model.load_state_dict(best_state)
print(f'Hoan tat sau {elapsed_minutes:.1f} phut. Best val AUC: {best_val_auc:.4f}')
print(f'Checkpoint: {checkpoint_path}')


## 5. Bieu do qua trinh huan luyen

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, metric, title in zip(axes, ['loss', 'accuracy', 'auc'], ['Loss', 'Accuracy', 'ROC AUC']):
    ax.plot(epochs_ran, history[f'train_{metric}'], marker='o', label='Train')
    ax.plot(epochs_ran, history[f'val_{metric}'], marker='o', label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('EfficientNet-B4 From Scratch training history', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Danh gia tren tap test: report, confusion matrix va ROC

In [ ]:
test_metrics = run_epoch(model, test_loader, criterion)
print(f"Test loss: {test_metrics['loss']:.4f} | Test accuracy: {test_metrics['accuracy']:.4f} | Test AUC: {test_metrics['auc']:.4f}")
print('\nClassification report:')
print(classification_report(test_metrics['labels'], test_metrics['predictions'], target_names=class_names, digits=4))

cm = confusion_matrix(test_metrics['labels'], test_metrics['predictions'])
fpr, tpr, _ = roc_curve(test_metrics['labels'], test_metrics['probs'])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
image = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].set_xticks(range(len(class_names)), labels=class_names)
axes[0].set_yticks(range(len(class_names)), labels=class_names)
threshold = cm.max() / 2
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        axes[0].text(col, row, cm[row, col], ha='center', va='center', color='white' if cm[row, col] > threshold else 'black')
axes[0].set_title('Confusion matrix - Test')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['auc']:.4f}")
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('ROC curve - Test')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'test_evaluation.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Hien thi mot so du doan

In [ ]:
def denormalize(image):
    mean = torch.tensor(NORMALIZE_MEAN).view(3, 1, 1)
    std = torch.tensor(NORMALIZE_STD).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0, 1)

images, labels = next(iter(test_loader))
with torch.no_grad():
    with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        probabilities = torch.softmax(model(images.to(DEVICE)), dim=1).cpu()
predictions = probabilities.argmax(dim=1)
show_count = min(8, len(images))
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for idx, ax in enumerate(axes.flatten()):
    if idx >= show_count:
        ax.axis('off')
        continue
    ax.imshow(denormalize(images[idx]).permute(1, 2, 0))
    actual = class_names[labels[idx]]
    predicted = class_names[predictions[idx]]
    confidence = probabilities[idx, predictions[idx]].item()
    color = 'green' if labels[idx] == predictions[idx] else 'red'
    ax.set_title(f'That: {actual}\nDoan: {predicted} ({confidence:.1%})', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('EfficientNet-B4 From Scratch - Du doan mau tren test')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=200, bbox_inches='tight')
plt.show()
